In [1]:
from nba_crunch.data_processing import enrich_free_throws, parse_clock
print(parse_clock('PT08M11.00S'))

491.0


In [2]:
from nba_api.stats.endpoints import playbyplayv3, leaguegamefinder
from nba_api.stats.static import teams
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [3]:
lakers = [t for t in teams.get_teams() if t['full_name'] == 'Los Angeles Lakers'][0]
lakers_id = lakers['id']
print(lakers_id)

1610612747


In [4]:
gamefinder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=lakers_id,
    season_nullable='2025-26',
    season_type_nullable='Regular Season',
    league_id_nullable='00'
)

games = gamefinder.get_data_frames()[0]
print(games.shape)


(82, 28)


In [5]:
games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,22025,1610612747,LAL,Los Angeles Lakers,0022501198,2026-04-12,LAL vs. UTA,W,240,131,51,93,0.548,15,34,0.441,14,17,0.824,11,38,49,37,12,6,15,20,24.0
1,22025,1610612747,LAL,Los Angeles Lakers,0022501185,2026-04-10,LAL vs. PHX,W,241,101,35,69,0.507,8,20,0.400,23,30,0.767,4,31,35,27,17,3,11,18,28.0
2,22025,1610612747,LAL,Los Angeles Lakers,0022501170,2026-04-09,LAL @ GSW,W,243,119,49,80,0.613,16,29,0.552,5,8,0.625,8,25,33,37,14,3,17,13,16.0
3,22025,1610612747,LAL,Los Angeles Lakers,0022501155,2026-04-07,LAL vs. OKC,L,242,87,32,73,0.438,9,26,0.346,14,31,0.452,8,30,38,26,7,3,17,12,-36.0
4,22025,1610612747,LAL,Los Angeles Lakers,0022501140,2026-04-05,LAL @ DAL,L,240,128,47,91,0.516,8,27,0.296,26,33,0.788,11,38,49,36,6,1,12,23,-6.0


In [6]:
game_id = '0022501024'
print(game_id)

0022501024


In [7]:
pbp = playbyplayv3.PlayByPlayV3(game_id=game_id).get_data_frames()[0]
print(pbp.shape)
print(pbp.columns.tolist())

(519, 24)
['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']


In [8]:
pbp.head()

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022501024,2,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (7:11 PM EST),period,start,1,0,1
1,0022501024,4,PT12M00.00S,1,1610612753,ORL,1628976,Carter Jr.,W. Carter Jr.,0,0,0,,0,,,0,h,Jump Ball Carter Jr. vs. Ayton: Tip to L. James,Jump Ball,,1,0,2
2,0022501024,7,PT11M46.00S,1,1610612747,LAL,1629028,Ayton,D. Ayton,-6,1,1,Made,1,0,2,2,v,Ayton 1' Alley Oop Dunk (2 PTS) (Reaves 1 AST),Made Shot,Alley Oop Dunk Shot,1,2,3
3,0022501024,9,PT11M27.00S,1,1610612753,ORL,1630217,Bane,D. Bane,0,0,0,,0,,,0,h,Bane Lost Ball Turnover (P1.T1),Turnover,Lost Ball,1,0,4
4,0022501024,9,PT11M27.00S,1,1610612747,LAL,2544,James,L. James,0,0,0,,0,,,0,v,L. James STEAL (1 STL),,,1,0,5


In [9]:
pbp['actionType'].value_counts()

actionType
Rebound           109
Missed Shot        95
Made Shot          75
Substitution       53
Free Throw         52
Foul               49
Turnover           30
                   28
Timeout            12
period              8
Violation           3
Instant Replay      2
Heave               2
Jump Ball           1
Name: count, dtype: int64

In [10]:
pbp[pbp['actionType'] == 'Free Throw']

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
18,0022501024,28,PT09M30.00S,1,1610612747,LAL,1630559,Reaves,A. Reaves,0,0,0,,0,6,7,13,v,Reaves Free Throw 1 of 2 (1 PTS),Free Throw,Free Throw 1 of 2,1,0,19
19,0022501024,29,PT09M30.00S,1,1610612747,LAL,1630559,Reaves,A. Reaves,0,0,0,,0,6,8,14,v,Reaves Free Throw 2 of 2 (2 PTS),Free Throw,Free Throw 2 of 2,1,0,20
21,0022501024,32,PT09M20.00S,1,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,7,8,15,h,Banchero Free Throw 1 of 2 (1 PTS),Free Throw,Free Throw 1 of 2,1,0,22
22,0022501024,33,PT09M20.00S,1,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,8,8,16,h,Banchero Free Throw 2 of 2 (2 PTS),Free Throw,Free Throw 2 of 2,1,0,23
28,0022501024,41,PT08M54.00S,1,1610612753,ORL,1641783,da Silva,T. da Silva,0,0,0,,0,9,10,19,h,da Silva Free Throw 1 of 2 (4 PTS),Free Throw,Free Throw 1 of 2,1,0,29
29,0022501024,42,PT08M54.00S,1,1610612753,ORL,1641783,da Silva,T. da Silva,0,0,0,,0,10,10,20,h,da Silva Free Throw 2 of 2 (5 PTS),Free Throw,Free Throw 2 of 2,1,0,30
102,0022501024,134,PT01M17.00S,1,1610612753,ORL,1629048,Bitadze,G. Bitadze,0,0,0,,0,24,37,61,h,Bitadze Free Throw 1 of 2 (1 PTS),Free Throw,Free Throw 1 of 2,1,0,103
104,0022501024,137,PT01M17.00S,1,1610612753,ORL,1629048,Bitadze,G. Bitadze,0,0,0,,0,25,37,62,h,Bitadze Free Throw 2 of 2 (2 PTS),Free Throw,Free Throw 2 of 2,1,0,105
115,0022501024,152,PT00M25.40S,1,1610612753,ORL,1631288,Cain,J. Cain,0,0,0,,0,28,37,65,h,Cain Free Throw 1 of 2 (3 PTS),Free Throw,Free Throw 1 of 2,1,0,116
117,0022501024,155,PT00M25.40S,1,1610612753,ORL,1631288,Cain,J. Cain,0,0,0,,0,,,0,h,MISS Cain Free Throw 2 of 2,Free Throw,Free Throw 2 of 2,1,0,118


In [11]:
print(pbp.columns.tolist())
print(pbp.dtypes)
print(pbp.shape)

['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']
gameId              str
actionNumber      int64
clock               str
period            int64
teamId            int64
teamTricode         str
personId          int64
playerName          str
playerNameI         str
xLegacy           int64
yLegacy           int64
shotDistance      int64
shotResult          str
isFieldGoal       int64
scoreHome           str
scoreAway           str
pointsTotal       int64
location            str
description         str
actionType          str
subType             str
videoAvailable    int64
shotValue         int64
actionId          int64
dtype: object
(519, 24)


In [12]:
fts = pbp[pbp['actionType'] == 'Free Throw']  # or whatever the right filter is
fts['shotResult'].value_counts()

shotResult
    52
Name: count, dtype: int64

In [13]:
fts[fts['period'] >= 4]

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
406,0022501024,568,PT09M01.00S,4,1610612747,LAL,1629637,Hayes,J. Hayes,0,0,0,,0,,,0,v,MISS Hayes Free Throw 1 of 2,Free Throw,Free Throw 1 of 2,1,0,407
409,0022501024,572,PT09M01.00S,4,1610612747,LAL,1629637,Hayes,J. Hayes,0,0,0,,0,,,0,v,MISS Hayes Free Throw 2 of 2,Free Throw,Free Throw 2 of 2,1,0,410
434,0022501024,606,PT06M31.00S,4,1610612747,LAL,1630559,Reaves,A. Reaves,0,0,0,,0,91,91,182,v,Reaves Free Throw 1 of 2 (20 PTS),Free Throw,Free Throw 1 of 2,1,0,435
435,0022501024,607,PT06M31.00S,4,1610612747,LAL,1630559,Reaves,A. Reaves,0,0,0,,0,91,92,183,v,Reaves Free Throw 2 of 2 (21 PTS),Free Throw,Free Throw 2 of 2,1,0,436
439,0022501024,614,PT06M20.00S,4,1610612753,ORL,1628976,Carter Jr.,W. Carter Jr.,0,0,0,,0,94,92,186,h,Carter Jr. Free Throw 1 of 1 (9 PTS),Free Throw,Free Throw 1 of 1,1,0,440
470,0022501024,649,PT03M18.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,97,95,192,h,Banchero Free Throw 1 of 2 (11 PTS),Free Throw,Free Throw 1 of 2,1,0,471
475,0022501024,656,PT03M18.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,98,95,193,h,Banchero Free Throw 2 of 2 (12 PTS),Free Throw,Free Throw 2 of 2,1,0,476
489,0022501024,677,PT00M50.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,103,99,202,h,Banchero Free Throw 1 of 2 (15 PTS),Free Throw,Free Throw 1 of 2,1,0,490
490,0022501024,678,PT00M50.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,104,99,203,h,Banchero Free Throw 2 of 2 (16 PTS),Free Throw,Free Throw 2 of 2,1,0,491
497,0022501024,686,PT00M05.50S,4,1610612747,LAL,1629028,Ayton,D. Ayton,0,0,0,,0,104,102,206,v,Ayton Free Throw 1 of 2 (9 PTS),Free Throw,Free Throw 1 of 2,1,0,498


## Define crunch time
1. Parse clock
2. Forward fill score
3. Compute margin

In [14]:
# 1 Parse clock
def parse_clock(clock_str):
    """
    Parse ISO 8601 duration string like 'PT08M11.00S' to total seconds remaining.
    
    Examples:
        parse_clock('PT08M11.00S') -> 491.0
        parse_clock('PT00M30.00S') -> 30.0
        parse_clock('PT12M00.00S') -> 720.0
    """
    # Strip the 'PT' prefix and split on M
    parts = clock_str.replace('PT', '').split('M')

    # Parse mins and secs and convert to float
    mins = float(parts[0])
    secs = float(parts[1].replace('S', ''))

    # Return total seconds (minutes * 60 + seconds)
    seconds = (mins * 60) + secs
    return seconds

pbp['scoreHome'] = pbp['scoreHome'].replace('', np.nan).ffill().fillna(0).astype(int)
pbp['scoreAway'] = pbp['scoreAway'].replace('', np.nan).ffill().fillna(0).astype(int)

fts = pbp[pbp['actionType'] == 'Free Throw'].copy()
fts['clock_seconds'] = fts['clock'].apply(parse_clock)
fts['made'] = ~fts['description'].str.startswith('MISS')



# import numpy as np

# fts['scoreHome'] = fts['scoreHome'].replace('', np.nan).ffill().fillna(0).astype(int)
# fts['scoreAway'] = fts['scoreAway'].replace('', np.nan).ffill().fillna(0).astype(int)

# fts['clock_seconds'] = fts['clock'].apply(parse_clock)
# fts['made'] = ~fts['description'].str.startswith('MISS')


In [15]:
fts[['description', 'scoreHome', 'scoreAway', 'made']].head(10)

,description,scoreHome,scoreAway,made
18,Reaves Free Throw 1 of 2 (1 PTS),6,7,True
19,Reaves Free Throw 2 of 2 (2 PTS),6,8,True
21,Banchero Free Throw 1 of 2 (1 PTS),7,8,True
22,Banchero Free Throw 2 of 2 (2 PTS),8,8,True
28,da Silva Free Throw 1 of 2 (4 PTS),9,10,True
29,da Silva Free Throw 2 of 2 (5 PTS),10,10,True
102,Bitadze Free Throw 1 of 2 (1 PTS),24,37,True
104,Bitadze Free Throw 2 of 2 (2 PTS),25,37,True
115,Cain Free Throw 1 of 2 (3 PTS),28,37,True
117,MISS Cain Free Throw 2 of 2,28,37,False


In [16]:
fts['margin'] = (fts['scoreHome'] - fts['scoreAway']).abs()
fts[['period', 'clock_seconds','description', 'scoreHome', 'scoreAway', 'margin']].head()

,period,clock_seconds,description,scoreHome,scoreAway,margin
18,1,570.0,Reaves Free Throw 1 of 2 (1 PTS),6,7,1
19,1,570.0,Reaves Free Throw 2 of 2 (2 PTS),6,8,2
21,1,560.0,Banchero Free Throw 1 of 2 (1 PTS),7,8,1
22,1,560.0,Banchero Free Throw 2 of 2 (2 PTS),8,8,0
28,1,534.0,da Silva Free Throw 1 of 2 (4 PTS),9,10,1


In [17]:
fts['is_crunch'] = (
    (fts['period'] >= 4) &
    (fts['clock_seconds'] <= 300) &
    (fts['margin'] <= 5)
)

fts[['period', 'clock_seconds','description', 'scoreHome', 'scoreAway', 'margin', 'is_crunch']].head()

,period,clock_seconds,description,scoreHome,scoreAway,margin,is_crunch
18,1,570.0,Reaves Free Throw 1 of 2 (1 PTS),6,7,1,False
19,1,570.0,Reaves Free Throw 2 of 2 (2 PTS),6,8,2,False
21,1,560.0,Banchero Free Throw 1 of 2 (1 PTS),7,8,1,False
22,1,560.0,Banchero Free Throw 2 of 2 (2 PTS),8,8,0,False
28,1,534.0,da Silva Free Throw 1 of 2 (4 PTS),9,10,1,False


In [18]:
fts['is_crunch'].sum()

np.int64(6)

In [19]:
close_games = games[games['PLUS_MINUS'].abs() <= 3]
close_games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'PLUS_MINUS']].head()

,GAME_ID,GAME_DATE,MATCHUP,WL,PLUS_MINUS
10,0022501038,2026-03-23,LAL @ DET,L,-3.0
11,0022501024,2026-03-21,LAL @ ORL,W,1.0
15,0022500974,2026-03-14,LAL vs. DEN,W,2.0
24,0022500855,2026-02-26,LAL @ PHX,L,-3.0
25,0022500840,2026-02-24,LAL vs. ORL,L,-1.0


In [20]:
import sys
sys.path.append('../src')  # so the notebook can find your src files
from data_processing import enrich_free_throws

# pull one game's pbp as before
fts = enrich_free_throws(pbp)
fts[fts['is_crunch']]

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId,clock_seconds,made,margin,is_crunch
470,0022501024,649,PT03M18.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,97,95,192,h,Banchero Free Throw 1 of 2 (11 PTS),Free Throw,Free Throw 1 of 2,1,0,471,198.0,True,2,True
475,0022501024,656,PT03M18.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,98,95,193,h,Banchero Free Throw 2 of 2 (12 PTS),Free Throw,Free Throw 2 of 2,1,0,476,198.0,True,3,True
489,0022501024,677,PT00M50.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,103,99,202,h,Banchero Free Throw 1 of 2 (15 PTS),Free Throw,Free Throw 1 of 2,1,0,490,50.0,True,4,True
490,0022501024,678,PT00M50.00S,4,1610612753,ORL,1631094,Banchero,P. Banchero,0,0,0,,0,104,99,203,h,Banchero Free Throw 2 of 2 (16 PTS),Free Throw,Free Throw 2 of 2,1,0,491,50.0,True,5,True
497,0022501024,686,PT00M05.50S,4,1610612747,LAL,1629028,Ayton,D. Ayton,0,0,0,,0,104,102,206,v,Ayton Free Throw 1 of 2 (9 PTS),Free Throw,Free Throw 1 of 2,1,0,498,5.5,True,2,True
500,0022501024,691,PT00M05.50S,4,1610612747,LAL,1629028,Ayton,D. Ayton,0,0,0,,0,104,102,0,v,MISS Ayton Free Throw 2 of 2,Free Throw,Free Throw 2 of 2,1,0,501,5.5,False,2,True


In [21]:
%load_ext autoreload
%autoreload 2

from nba_crunch.data_processing import enrich_free_throws, parse_clock
print(parse_clock('PT08M11.00S'))  # should print 491.0

491.0


In [22]:
from nba_api.stats.endpoints import playbyplayv3
from nba_crunch.data_processing import enrich_free_throws

pbp = playbyplayv3.PlayByPlayV3(game_id='0022501024').get_data_frames()[0]
fts = enrich_free_throws(pbp)
print(f"Total FTs: {len(fts)}")
print(f"Crunch FTs: {fts['is_crunch'].sum()}")
fts[fts['is_crunch']][['period', 'clock_seconds', 'margin', 'description', 'made']]

Total FTs: 52
Crunch FTs: 6


,period,clock_seconds,margin,description,made
470,4,198.0,2,Banchero Free Throw 1 of 2 (11 PTS),True
475,4,198.0,3,Banchero Free Throw 2 of 2 (12 PTS),True
489,4,50.0,4,Banchero Free Throw 1 of 2 (15 PTS),True
490,4,50.0,5,Banchero Free Throw 2 of 2 (16 PTS),True
497,4,5.5,2,Ayton Free Throw 1 of 2 (9 PTS),True
500,4,5.5,2,MISS Ayton Free Throw 2 of 2,False
